In [19]:
import pandas as pd
import pymysql
from pymysql import Error


# Lee el CSV
df = pd.read_csv('PS_20174392719_1491204439457_log.csv') #ruta del archivo CSV para leer 
print(f"Filas: {len(df)}")
print(f"Columnas: {df.columns.tolist()}")
print(df.head())

Filas: 6362620
Columnas: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  

# DETECCIÓN DE FRAUDE Y AML

## ESTRUCTURA DE LAS BASES DE DATOS

### 1. Tabla Original: `transacciones_paysim`
**Descripción:** Tabla cruda con 6.3 millones de transacciones bancarias simuladas (Kaggle PaySim).
**Propósito:** Contiene los datos transaccionales sin procesar.

| Columna | Tipo | Descripción | Uso en el Proyecto |
| :--- | :--- | :--- | :--- |
| `id` | INT | Identificador único de la transacción. | Llave primaria. |
| `step` | INT | Unidad de tiempo (1 step = 1 hora). | Para análisis temporal (no usado directamente como feature). |
| `type` | VARCHAR | Tipo de transacción: `PAYMENT`, `TRANSFER`, `CASH_OUT`, `CASH_IN`, `DEBIT`. | **Feature:** Se transforma en One-Hot Encoding y en `tipo_riesgoso`. |
| `amount` | DECIMAL | Monto de la transacción en moneda local. | **Feature:** Base de `monto_alto`, `monto_relativo`, `monto_riesgo_score`. |
| `nameOrig` | VARCHAR | Cliente que inicia la transacción (Origen). | Identificador (no se usa como feature para evitar Data Leakage). |
| `oldbalanceOrg` | DECIMAL | Saldo del cliente **antes** de la transacción. | **Feature:** Base de `ratio_gasto_saldo` y `cambio_saldo`. |
| `newbalanceOrig` | DECIMAL | Saldo del cliente **después** de la transacción. | **Feature:** Base de `saldo_cero` y `cambio_saldo`. |
| `nameDest` | VARCHAR | Cliente que recibe la transacción (Destino). | Identificador (no se usa como feature). |
| `oldbalanceDest` | DECIMAL | Saldo del destinatario **antes** de la transacción. | **Feature:** Base de `destino_saldo_cero`. |
| `newbalanceDest` | DECIMAL | Saldo del destinatario **después** de la transacción. | **Feature:** Base de `cambio_saldo_dest` (no implementada en V4). |
| `isFraud` | TINYINT | **Variable Objetivo.** 1 = Fraude, 0 = Normal. | **Target** del modelo. |
| `isFlaggedFraud` | TINYINT | 1 = Transacción marcada por reglas AML internas. | No se usa como feature (causaría Data Leakage). |

---

In [21]:
# Conecta a MySQL usando los permisos del servidor local
try:
    conn = pymysql.connect(
        host='localhost', #host del servidor
        user='usr', #usuario del servidor 
        password='contraseña', #contraseña del servidor sql
        database='banco_aml',
        unix_socket='ruta', #ruta del socker
        autocommit=True,
        charset='char' #charset del servidor
    ) 
    cursor = conn.cursor()
    print("Conectado a MySQL")

  # Crea la tabla (si no existe)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS transacciones_paysim (
            id INT AUTO_INCREMENT PRIMARY KEY,
            step INT NOT NULL,
            type VARCHAR(20) NOT NULL,
            amount DECIMAL(15,2) NOT NULL,
            nameOrig VARCHAR(50) NOT NULL,
            oldbalanceOrg DECIMAL(15,2) DEFAULT 0,
            newbalanceOrig DECIMAL(15,2) DEFAULT 0,
            nameDest VARCHAR(50) NOT NULL,
            oldbalanceDest DECIMAL(15,2) DEFAULT 0,
            newbalanceDest DECIMAL(15,2) DEFAULT 0,
            isFraud TINYINT(1) DEFAULT 0,
            isFlaggedFraud TINYINT(1) DEFAULT 0
        )
    """)
    print("Tabla creada") 
    
    # Inserta los datos en batches (más rápido)
    print("Insertando datos...")
    batch_size = 10000 #longitud del lote de filas que se procesan a la vex
    total_rows = len(df) #hacer el proceso para la cantidad de filas en el csv
    
    for i in range(0, total_rows, batch_size):
        batch = df.iloc[i:i+batch_size]
        values = []
        for _, row in batch.iterrows(): #aplicando el proceso para cada columna
            values.append((
                int(row['step']),
                row['type'],
                float(row['amount']),
                row['nameOrig'],
                float(row['oldbalanceOrg']),
                float(row['newbalanceOrig']),
                row['nameDest'],
                float(row['oldbalanceDest']),
                float(row['newbalanceDest']),
                int(row['isFraud']),
                int(row['isFlaggedFraud'])
            ))
        
        cursor.executemany(""" 
            INSERT INTO transacciones_paysim 
            (step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig, 
             nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        """, values) #definimos en que fila del sql lo vamos a insertar
        conn.commit()
        print(f"Insertadas {min(i+batch_size, total_rows)}/{total_rows} filas")
    
    print(f"Datos insertados: {total_rows} filas")
    
    # Verifica
    cursor.execute("SELECT COUNT(*) FROM transacciones_paysim") #consultamos la cantidad de filas en la database
    count = cursor.fetchone()[0]
    print(f"Total en tabla: {count}")
    
    cursor.close()
    conn.close()
    
except Error as e:
    print(f"Error: {e}")

✅ Conectado a MySQL
✅ Tabla creada
Insertando datos...
Insertadas 10000/6362620 filas
Insertadas 20000/6362620 filas
Insertadas 30000/6362620 filas
Insertadas 40000/6362620 filas
Insertadas 50000/6362620 filas
Insertadas 60000/6362620 filas
Insertadas 70000/6362620 filas
Insertadas 80000/6362620 filas
Insertadas 90000/6362620 filas
Insertadas 100000/6362620 filas
Insertadas 110000/6362620 filas
Insertadas 120000/6362620 filas
Insertadas 130000/6362620 filas
Insertadas 140000/6362620 filas
Insertadas 150000/6362620 filas
Insertadas 160000/6362620 filas
Insertadas 170000/6362620 filas
Insertadas 180000/6362620 filas
Insertadas 190000/6362620 filas
Insertadas 200000/6362620 filas
Insertadas 210000/6362620 filas
Insertadas 220000/6362620 filas
Insertadas 230000/6362620 filas
Insertadas 240000/6362620 filas
Insertadas 250000/6362620 filas
Insertadas 260000/6362620 filas
Insertadas 270000/6362620 filas
Insertadas 280000/6362620 filas
Insertadas 290000/6362620 filas
Insertadas 300000/6362620 